# Experimentul 7: VQC cu ansatz redus (3 → 2 repetări), pe ZZ feature map ×1

**O singură modificare față de Experimentul 3: `ANSATZ_REPS = 2`. Antrenare fără zgomot, evaluare pe validation la p = 0, 0.001, 0.01, 0.05.**

Experimentul 3 (ZZ ×1, ansatz ×3) este soluția principală, deja confirmată pe test în Experimentul 5. Rezultatele de test au fost consultate în dezvoltarea proiectului, deci **Experimentul 7 este exploratoriu**: se evaluează doar pe validation, iar testul nu se folosește.

## Ipoteze (de testat, nu presupuse)
- **H1:** cu o repetare mai puțin a ansatz-ului (56 de porți native, 18 CNOT, 74 de locuri de zgomot față de 67 / 21 / 88) circuitul acumulează mai puțin zgomot, deci AUPRC și ρ la p = 0.01 și 0.05 scad mai puțin, iar costul de antrenare pe hardware scade (24 de parametri în loc de 32).
- **H0 / alternativă:** un ansatz mai scurt este mai puțin expresiv: modelul pierde la p = 0 mai mult decât câștigă în robustețe. Zgomotul din ansatz e oricum mic față de cel din feature map, deci câștigul de robustețe poate fi neglijabil.

## Ce ținem fix (identic cu Experimentul 3)
Splitul comun (`split_indices.npz`), `undersample(X_train, y_train, seed)` 1:1, `AngleEncoder(4, seed)` fitat pe setul 1:1, 4 qubiți, ZZ feature map ×1, aceeași structură de ansatz (RY + RZ pe fiecare qubit, scară liniară de CNOT), readout pe qubitul 0, Adam, lr 0.05, batch 32, BCE, 30 de epoci, epoca aleasă după AUPRC pe validation fără zgomot, seeds `[0, 1, 2, 3, 4]`, modelul de zgomot oficial (`noisy_qnode`), pragul max-F1 pe validation la același p.

**Comparație pereche.** Pentru un seed dat, Exp. 3 și Exp. 7 au același set de antrenare, același encoder și aceeași ordine a mini-batch-urilor. `init_weights(seed)` trage din același generator: ponderile inițiale ale Exp. 7 (forma 3 × 4 × 2) sunt exact primele 3 straturi ale ponderilor inițiale ale Exp. 3 (forma 4 × 4 × 2); verificăm asta mai jos.

## 0. Verificarea configurației oficiale (notebookul Task 1 și PDF-ul Task 2)

Citite din `notebooks/task1_guided_vqc_fraud_detection.ipynb` și `docs/Fraud_on_a_Quantum_Budget_Task.pdf`; valorile marcate cu ✓ sunt verificate și în cod, mai jos.

| | Sursa oficială | Exp. 3 | Exp. 7 |
|---|---|---|---|
| Tranzacții / fraude | 284,807 / 492 (check 2.1) ✓ | idem | idem |
| Split | 60/20/20 stratificat, seed 42, `split_indices.npz`; test 56,962 rânduri, 98 fraude; validation 99 fraude (check 4.1) ✓ | idem | idem |
| Undersampling | toate fraudele din train + tot atâtea legitime, fără înlocuire, `rng = default_rng(seed)`, apoi permutare (ex. 4.2) | idem | idem |
| Exemple de antrenare | 295 fraude + 295 legitime = **590** (Partea 4.3 și ledger-ul din Partea 8; PDF: „30 epochs on 590 balanced samples”) ✓ | 590 | 590 |
| Encoder | StandardScaler pe Time/Amount → PCA(4) → MinMax la [0, π], fitat pe setul echilibrat, clip la [0, π] | idem | idem |
| Qubiți | 4 | 4 | 4 |
| ZZ feature map | H + RZ(2xᵢ) pe fiecare qubit, apoi pentru fiecare pereche i < j (6 perechi): CNOT, RZ(2(π − xᵢ)(π − xⱼ)), CNOT; **2 repetări** | **1 repetare** | 1 repetare |
| Ansatz | EfficientSU2-style: RY + RZ pe fiecare qubit, CNOT 0→1, 1→2, 2→3, × `ANSATZ_REPS`, plus stratul final de rotații; parametri = (`ANSATZ_REPS` + 1) × 4 × 2; **3 repetări, 32 de parametri** | 3, 32 | **2, 24** |
| Readout | (1 − ⟨Z₀⟩) / 2 | idem | idem |
| Antrenare | Adam, lr 0.05, batch 32, BCELoss, 30 de epoci, epoca cu AUPRC maxim pe validation (strict >) | idem | idem |
| Zgomot | decompoziție în {RX, RY, RZ, Rot, H, CNOT, CZ}, `DepolarizingChannel(p)` după fiecare poartă pe fiecare qubit, `default.mixed`; p = 0.001, 0.01, 0.05 | idem | idem |
| Evaluare Task 2 | prag max-F1 pe validation la același p; AUPRC (principală, p = 0.01 și 0.05), AUROC, Spearman ρ, precision, recall, F1, MCC; medie ± std pe ≥ 5 seeds | idem, pe validation | idem, pe validation |
| Cost | antrenare = epoci × 590 × (1 + 2P), validare = epoci × n_val, 1024 shots / execuție; `qml.Tracker` pentru parameter shift | idem | idem |

**Diferențe față de notebookul oficial, deja prezente în Exp. 3 (păstrate identic):**
1. `FEATURE_MAP_REPS = 1` în loc de 2: modificarea intenționată a Exp. 3.
2. Amestecarea mini-batch-urilor folosește un `torch.Generator().manual_seed(seed)` dedicat, în loc de generatorul global după `set_seed(seed)`. Verificăm mai jos că produce **aceleași permutări** ca varianta oficială, deci nu e o diferență de metodă.
3. Evaluarea se face pe validation, nu pe test (testul nu se folosește în această etapă).

Nu există neconcordanțe care să schimbe metodologia: Exp. 7 reproduce Exp. 3 cu o singură modificare, `ANSATZ_REPS = 2`.

## 1. Configurație și imports

**Seeds.** `set_seed` fixează numpy și torch; amestecarea folosește un `torch.Generator` dedicat (ca în Exp. 3). Simulatoarele PennyLane rulează cu `shots=None` (valori exacte); singura excepție este numărarea execuțiilor parameter-shift cu `qml.Tracker`, pe un dispozitiv cu shots și `seed` fix.

In [ ]:
import json
import math
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pennylane as qml
import sklearn
import torch
from IPython.display import display
from scipy.stats import spearmanr
from sklearn.decomposition import PCA
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# ---- singura modificare față de Exp. 3 ----
ANSATZ_REPS = 2               # Exp. 3: 3
EXP3_ANSATZ_REPS = 3          # doar pentru comparația structurală

# ---- fix, identic cu Experimentul 3 ----
SPLIT_SEED = 42
N_QUBITS = 4
FEATURE_MAP_REPS = 1
EPOCHS = 30
BATCH_SIZE = 32
LEARNING_RATE = 0.05
SHOTS_PER_EXECUTION = 1024    # doar pentru cost ledger
NOISE_LEVELS = [0.001, 0.01, 0.05]
NATIVE_GATES = {"RX", "RY", "RZ", "Rot", "Hadamard", "CNOT", "CZ"}
SEEDS = [0, 1, 2, 3, 4]
P_ALL = [0.0, *NOISE_LEVELS]

EXPERIMENT_NAME = "reduced_ansatz"
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "notebooks").is_dir() and (p / "src").is_dir())
RESULTS_DIR = REPO_ROOT / "results" / "result_exp7"
WEIGHTS_DIR = RESULTS_DIR / "weights"
PLOTS_DIR = RESULTS_DIR / "plots"
EXP3_DIR = REPO_ROOT / "results" / "result_exp3"   # doar citit
SPLIT_PATH = REPO_ROOT / "split_indices.npz"

EXP3_COLOR, EXP7_COLOR, SEED_DOT_COLOR = "#2a78d6", "#eb6834", "#8a8f98"
EXP3_LABEL, EXP7_LABEL = "Exp. 3 (ansatz ×3)", "Exp. 7 (ansatz ×2)"
plt.rcParams.update({"axes.grid": True, "grid.alpha": 0.25, "axes.spines.top": False, "axes.spines.right": False,
                     "lines.linewidth": 2})


def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)


set_seed(SPLIT_SEED)
print(f"ANSATZ_REPS = {ANSATZ_REPS} (Exp. 3: {EXP3_ANSATZ_REPS})   FEATURE_MAP_REPS = {FEATURE_MAP_REPS}   seeds: {SEEDS}   p: {P_ALL}")
print("pennylane", qml.__version__, "| torch", torch.__version__, "| sklearn", sklearn.__version__)
print("rezultate:", RESULTS_DIR.relative_to(REPO_ROOT))

## 2. Datele, splitul comun și dimensiunea setului de antrenare

Splitul se încarcă din `split_indices.npz`, în ordinea salvată (ordinea rândurilor din training contează pentru `undersample`), și se verifică față de reconstrucția oficială (Partea 4). Etichetele întregului dataset sunt folosite doar pentru a reconstrui splitul stratificat; **features-urile testului nu se încarcă**, iar etichetele testului nu se citesc.

Apoi verificăm, pentru fiecare seed, numărul de exemple înainte și după undersampling.

In [ ]:
DATA_PATH = REPO_ROOT / "data" / "creditcard.csv"
for path in [DATA_PATH, SPLIT_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"fișier obligatoriu lipsă: {path}")
df = pd.read_csv(DATA_PATH)
df["Class"] = df["Class"].astype(int)
assert df.shape == (284807, 31) and "Time" in df.columns and df.index.equals(pd.RangeIndex(len(df)))
assert (int((df["Class"] == 1).sum()), int((df["Class"] == 0).sum())) == (492, 284315)   # check 2.1 oficial

X_all, y_all = df.drop(columns="Class"), df["Class"].to_numpy()
split = np.load(SPLIT_PATH)
train_idx, val_idx, test_idx = split["train"], split["val"], split["test"]

_trval, _test = train_test_split(np.arange(len(df)), test_size=0.2, stratify=y_all, random_state=SPLIT_SEED)
_train, _val = train_test_split(_trval, test_size=0.25, stratify=y_all[_trval], random_state=SPLIT_SEED)
assert np.array_equal(_train, train_idx) and np.array_equal(_val, val_idx) and np.array_equal(_test, test_idx), "split_indices.npz diferă de splitul oficial"
assert not (set(train_idx) & set(val_idx)) and not (set(train_idx) & set(test_idx)) and not (set(val_idx) & set(test_idx))

X_train, y_train = X_all.loc[train_idx], y_all[train_idx]
X_val, y_val = X_all.loc[val_idx], y_all[val_idx]
assert (int(y_train.sum()), int(y_val.sum())) == (295, 99) and len(test_idx) == 56962
del X_all, y_all, df, _trval, _test, _train, _val   # testul nu rămâne în memorie

print(f"split: {SPLIT_PATH.name} (identic cu reconstrucția oficială)")
print(f"train {len(train_idx):,} ({int(y_train.sum())} fraude)   validation {len(val_idx):,} ({int(y_val.sum())} fraude)   test {len(test_idx):,} rânduri (neîncărcat)")

## 3. Funcțiile din notebookul oficial

Identice cu cele din Exp. 3 (golurile oficiale completate). `ansatz` și `init_weights` citesc `ANSATZ_REPS`, deci aici construiesc 2 repetări și 24 de parametri.

In [ ]:
def undersample(X_part, y_part, seed):
    rng = np.random.default_rng(seed)
    fraud = np.flatnonzero(y_part == 1)
    legit = rng.choice(np.flatnonzero(y_part == 0), size=len(fraud), replace=False)
    keep = rng.permutation(np.concatenate([fraud, legit]))
    return X_part.iloc[keep], y_part[keep]


class AngleEncoder:
    raw_columns = ["Time", "Amount"]

    def __init__(self, n_components, seed):
        self.scaler = StandardScaler()
        self.pca = PCA(n_components=n_components, random_state=seed)
        self.angles = MinMaxScaler(feature_range=(0, np.pi))

    def _scaled(self, frame):
        out = frame.copy()
        out[self.raw_columns] = self.scaler.transform(frame[self.raw_columns])
        return out.to_numpy()

    def fit(self, frame):
        self.scaler.fit(frame[self.raw_columns])
        self.angles.fit(self.pca.fit_transform(self._scaled(frame)))
        return self

    def transform(self, frame):
        return np.clip(self.angles.transform(self.pca.transform(self._scaled(frame))), 0, np.pi)


def feature_map(x):
    """ZZ feature map. x holds N_QUBITS angles in [0, pi], or a batch of them (shape (..., N_QUBITS))."""
    for _ in range(FEATURE_MAP_REPS):
        for i in range(N_QUBITS):
            qml.Hadamard(wires=i)
            qml.RZ(2 * x[..., i], wires=i)
        for i in range(N_QUBITS):
            for j in range(i + 1, N_QUBITS):
                qml.CNOT(wires=[i, j])
                qml.RZ(2 * (np.pi - x[..., i]) * (np.pi - x[..., j]), wires=j)
                qml.CNOT(wires=[i, j])


def ansatz(weights):
    """EfficientSU2-style ansatz; weights has shape (ANSATZ_REPS + 1, N_QUBITS, 2)."""
    for layer in range(ANSATZ_REPS + 1):
        for i in range(N_QUBITS):
            qml.RY(weights[layer, i, 0], wires=i)
            qml.RZ(weights[layer, i, 1], wires=i)
        if layer < ANSATZ_REPS:
            for i in range(N_QUBITS - 1):
                qml.CNOT(wires=[i, i + 1])


def vqc(x, weights):
    feature_map(x)
    ansatz(weights)
    return qml.expval(qml.PauliZ(0))


device = qml.device("default.qubit", wires=N_QUBITS)   # shots=None: valori de așteptare exacte
circuit = qml.QNode(vqc, device, interface="torch", diff_method="backprop")


def fraud_probability(x, weights):
    return (1 - circuit(x, weights)) / 2


def init_weights(seed):
    generator = torch.Generator().manual_seed(seed)
    return torch.nn.Parameter(2 * np.pi * torch.rand(ANSATZ_REPS + 1, N_QUBITS, 2, generator=generator, dtype=torch.float64))


def predict(Z, weights, model=None, chunk=20000):
    model = model or fraud_probability
    X_t = torch.tensor(Z)
    with torch.no_grad():
        return torch.cat([model(X_t[k:k + chunk], weights) for k in range(0, len(X_t), chunk)]).numpy()


def train_vqc_run(Z_tr, y_tr, Z_valid, y_valid, initial_weights, seed, epochs=EPOCHS):
    """Bucla din Exp. 3; în plus numără pașii Adam și timpul pe epocă."""
    set_seed(seed)
    shuffle_generator = torch.Generator().manual_seed(seed)
    weights = torch.nn.Parameter(initial_weights.detach().clone())
    optimizer = torch.optim.Adam([weights], lr=LEARNING_RATE)
    loss_fn = torch.nn.BCELoss()
    X_t = torch.tensor(Z_tr)
    y_t = torch.tensor(y_tr, dtype=torch.float64)
    history, best_ap, best_weights, best_epoch, steps = [], -1.0, None, None, 0
    start = time.perf_counter()
    for epoch in range(1, epochs + 1):
        t_epoch = time.perf_counter()
        order = torch.randperm(len(X_t), generator=shuffle_generator)
        losses = []
        for k in range(0, len(X_t), BATCH_SIZE):
            idx = order[k:k + BATCH_SIZE]
            optimizer.zero_grad()
            loss = loss_fn(fraud_probability(X_t[idx], weights), y_t[idx])
            loss.backward()
            optimizer.step()
            steps += 1
            losses.append(loss.item())
        val_ap = average_precision_score(y_valid, predict(Z_valid, weights))
        history.append({"epoch": epoch, "train loss": float(np.mean(losses)), "validation AUPRC": float(val_ap),
                        "timp epocă [s]": time.perf_counter() - t_epoch})
        if val_ap > best_ap:
            best_ap, best_weights, best_epoch = val_ap, weights.detach().clone(), epoch
    return {"weights": best_weights, "history": history, "best_epoch": best_epoch, "best_val_ap": float(best_ap),
            "adam_steps": steps, "time_train_s": time.perf_counter() - start}


def choose_threshold(y_true, scores):
    """The threshold that maximises F1 on (y_true, scores)."""
    precision, recall, thresholds = precision_recall_curve(y_true, scores)
    f1 = 2 * precision * recall / np.clip(precision + recall, 1e-12, None)
    return thresholds[np.argmax(f1[:-1])]


def evaluate(y_true, scores, threshold):
    predicted = (scores >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, predicted, labels=[0, 1]).ravel()
    return {
        "AUPRC": average_precision_score(y_true, scores),
        "AUROC": roc_auc_score(y_true, scores),
        "precision": precision_score(y_true, predicted, zero_division=0),
        "recall": recall_score(y_true, predicted, zero_division=0),
        "F1": f1_score(y_true, predicted, zero_division=0),
        "MCC": matthews_corrcoef(y_true, predicted),
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "TN": tn,
    }


def noisy_qnode(circuit_fn, p, n_wires=N_QUBITS):
    """QNode of circuit_fn under the challenge's noise model: native gates, then a depolarizing channel of
    probability p after every gate, on each of its qubits. Differentiable with backprop."""
    noisy_device = qml.noise.insert(qml.device("default.mixed", wires=n_wires), qml.DepolarizingChannel, p, position="all")
    qnode = qml.QNode(circuit_fn, noisy_device, interface="torch", diff_method="backprop")
    return qml.transforms.decompose(qnode, gate_set=NATIVE_GATES)


def noisy_predict(Z, weights, p, circuit_fn=vqc, readout=lambda expval: (1 - expval) / 2, n_wires=N_QUBITS, chunk=2000):
    """Fraud scores of the transactions Z when circuit_fn runs under depolarizing noise p."""
    noisy_circuit = noisy_qnode(circuit_fn, p, n_wires)
    X_t = torch.tensor(Z)
    with torch.no_grad():
        return torch.cat([readout(noisy_circuit(X_t[k:k + chunk], weights)) for k in range(0, len(X_t), chunk)]).numpy()


def noise_locations(circuit_fn, *args, n_wires=N_QUBITS):
    """Number of depolarizing channels the noise model inserts: one per qubit of every native gate."""
    qnode = qml.transforms.decompose(qml.QNode(circuit_fn, qml.device("default.qubit", wires=n_wires)), gate_set=NATIVE_GATES)
    return sum(size * count for size, count in qml.specs(qnode)(*args)["resources"].gate_sizes.items())

### Verificări: setul de antrenare (590 per seed), amestecarea, inițializarea

- Pentru fiecare seed: rânduri și fraude **înainte** de undersampling (train complet) și **după** (setul 1:1); validation rămâne neatins (56,962 rânduri, 99 fraude).
- Amestecarea cu generator dedicat produce aceleași permutări ca varianta oficială (`set_seed(seed)` + `torch.randperm` global).
- Ponderile inițiale ale Exp. 7 sunt primele 3 straturi ale celor din Exp. 3 (citite din `result_exp3/weights`).

In [ ]:
balance_rows = []
for seed in SEEDS:
    X_s, y_s = undersample(X_train, y_train, seed)
    assert len(y_s) == 2 * y_train.sum() and y_s.sum() == y_train.sum() and X_s.index.is_unique and set(X_s.index) <= set(X_train.index)
    balance_rows.append({"seed": seed, "train înainte: rânduri": len(y_train), "train înainte: fraude": int(y_train.sum()),
                         "train înainte: legitime": int((y_train == 0).sum()), "după undersampling: fraude": int(y_s.sum()),
                         "după undersampling: legitime": int((y_s == 0).sum()), "după undersampling: total": len(y_s),
                         "validation: rânduri": len(y_val), "validation: fraude": int(y_val.sum())})
balance = pd.DataFrame(balance_rows).set_index("seed")
assert (balance["după undersampling: total"] == 590).all() and (balance["după undersampling: fraude"] == 295).all()
print("✅ fiecare seed: 295 fraude + 295 legitime = 590 exemple de antrenare; validation cu distribuția originală")
display(balance)

# amestecarea: generator dedicat == generatorul global după set_seed (varianta oficială)
for seed in SEEDS:
    g = torch.Generator().manual_seed(seed)
    set_seed(seed)
    assert all(torch.equal(torch.randperm(590, generator=g), torch.randperm(590)) for _ in range(EPOCHS))
print("✅ permutările mini-batch-urilor coincid cu cele ale buclei oficiale (30 de epoci, toate seed-urile)")

# inițializarea: primele 3 straturi ale Exp. 3
for seed in SEEDS:
    exp3_init = torch.load(EXP3_DIR / "weights" / f"seed_{seed}.pt", weights_only=True)["initial_weights"]
    assert exp3_init.shape == (EXP3_ANSATZ_REPS + 1, N_QUBITS, 2)
    assert torch.equal(init_weights(seed).detach(), exp3_init[:ANSATZ_REPS + 1])
print(f"✅ init_weights(seed) pentru Exp. 7 = primele {ANSATZ_REPS + 1} straturi ale ponderilor inițiale din Exp. 3")

### Checks oficiale și structura circuitului: Exp. 3 vs Exp. 7

Checks 4.2, 9.1, 6.1 și 10.1 din notebookul oficial; check 6.2 (probabilitățile de referință) e definit doar pentru `ANSATZ_REPS = 3`, așa că pentru 2 repetări verificăm numărul de porți al ansatz-ului.

Structura **ambelor** circuite e măsurată cu aceleași unelte, schimbând temporar `ANSATZ_REPS`:
- porți native, CNOT, adâncime (`qml.specs` după decompoziția în `NATIVE_GATES`), locuri de zgomot (`noise_locations`), comparate cu formula așteptată;
- execuțiile parameter-shift **măsurate** cu `qml.Tracker` pe un mini-batch de 32 (ca în Partea 8): PennyLane nu deplasează parametrii care nu pot ajunge la qubitul 0;
- parametrii care **influențează efectiv** măsurătoarea: cei cu gradient nenul (backprop) pe 256 de exemple, cu ponderi aleatoare. Diferența față de Tracker este rotația RZ finală de pe qubitul 0, care comută cu Z și are gradient mereu 0 (Partea 8).

Valorile pentru Exp. 3 se compară cu `result_exp3/circuit_structure.csv`.

In [ ]:
X_bal_check, y_bal_check = undersample(X_train, y_train, SPLIT_SEED)   # 4.2
assert len(y_bal_check) == 2 * y_train.sum() and y_bal_check.sum() == y_train.sum() and X_bal_check.index.is_unique

rng_check = np.random.default_rng(3)   # 9.1
y_c = rng_check.integers(0, 2, 300)
s_c = np.clip(0.3 * y_c + rng_check.random(300), 0, 1)
best_f1 = max(f1_score(y_c, (s_c >= t).astype(int)) for t in np.unique(s_c))
assert np.isclose(f1_score(y_c, (s_c >= choose_threshold(y_c, s_c)).astype(int)), best_f1)

dev4 = qml.device("default.qubit", wires=N_QUBITS)


def state_of(circuit_fn, *args):
    def full():
        circuit_fn(*args)
        return qml.state()
    return qml.QNode(full, dev4)()


def gate_counts(circuit_fn, *args):
    def full():
        circuit_fn(*args)
        return qml.state()
    return qml.specs(qml.transforms.decompose(qml.QNode(full, dev4), gate_set=NATIVE_GATES))()["resources"].gate_types


def feature_map_reference(x):
    for _ in range(FEATURE_MAP_REPS):
        for i in range(N_QUBITS):
            qml.Hadamard(wires=i)
            qml.RZ(2 * x[i], wires=i)
        for i in range(N_QUBITS):
            for j in range(i + 1, N_QUBITS):
                qml.IsingZZ(2 * (np.pi - x[i]) * (np.pi - x[j]), wires=[i, j])


Z_check = AngleEncoder(N_QUBITS, SPLIT_SEED).fit(X_bal_check).transform(X_bal_check)
for x_check in Z_check[:3]:   # 6.1
    assert np.isclose(abs(np.vdot(state_of(feature_map_reference, x_check), state_of(feature_map, x_check))) ** 2, 1.0)

w_check = torch.linspace(0, 3, (ANSATZ_REPS + 1) * N_QUBITS * 2, dtype=torch.float64).reshape(ANSATZ_REPS + 1, N_QUBITS, 2)   # 6.2 (numărătoare)
counts = gate_counts(ansatz, w_check)
assert counts["RY"] == counts["RZ"] == (ANSATZ_REPS + 1) * N_QUBITS and counts["CNOT"] == ANSATZ_REPS * (N_QUBITS - 1)


@qml.qnode(qml.device("default.mixed", wires=1))   # 10.1
def noisy_ry(theta, p):
    qml.RY(theta, wires=0)
    qml.DepolarizingChannel(p, wires=0)
    return qml.expval(qml.PauliZ(0))


for theta, p in [(0.3, 0.0), (1.0, 0.1), (2.5, 0.3), (0.7, 0.75)]:
    assert np.isclose(noisy_ry(theta, p), (1 - 4 * p / 3) * np.cos(theta))
print("✅ Checks 4.2, 9.1, 6.1, 6.2 (numărătoare), 10.1: trecute")


def expected_structure(ansatz_reps):
    """Numărătoarea așteptată, din construcția circuitului: feature map + ansatz."""
    pairs = N_QUBITS * (N_QUBITS - 1) // 2
    fm_gates, fm_cnots = FEATURE_MAP_REPS * (2 * N_QUBITS + 3 * pairs), FEATURE_MAP_REPS * 2 * pairs
    an_gates, an_cnots = (ansatz_reps + 1) * N_QUBITS * 2 + ansatz_reps * (N_QUBITS - 1), ansatz_reps * (N_QUBITS - 1)
    return {"porți native": fm_gates + an_gates, "CNOT": fm_cnots + an_cnots, "locuri de zgomot": fm_gates + an_gates + fm_cnots + an_cnots,
            "parametri": (ansatz_reps + 1) * N_QUBITS * 2}


def circuit_structure(ansatz_reps, x, batch_x, batch_y, grad_x):
    """Structura, execuțiile parameter-shift și parametrii efectivi ai VQC-ului cu `ansatz_reps` repetări ale ansatz-ului."""
    global ANSATZ_REPS
    saved_reps, ANSATZ_REPS = ANSATZ_REPS, ansatz_reps   # schimbare temporară, restaurată în finally
    try:
        w_probe = init_weights(SEEDS[0])
        counts = gate_counts(vqc, x, w_probe)
        resources = qml.specs(qml.transforms.decompose(qml.QNode(vqc, dev4), gate_set=NATIVE_GATES))(x, w_probe)["resources"]
        shot_device = qml.device("default.qubit", wires=N_QUBITS, seed=SPLIT_SEED)
        hardware_like = qml.set_shots(qml.QNode(vqc, shot_device, interface="torch", diff_method="parameter-shift"), shots=SHOTS_PER_EXECUTION)
        with qml.Tracker(shot_device) as tracker:
            loss = torch.nn.BCELoss()((1 - hardware_like(batch_x, w_probe)) / 2, batch_y)
            loss.backward()
        exec_tracker = tracker.totals["executions"] / len(batch_x)
        w_grad = init_weights(SEEDS[0])
        fraud_probability(grad_x, w_grad).sum().backward()
        effective = int((w_grad.grad.abs() > 1e-10).sum())
        return {
            "repetări ansatz": ansatz_reps, "porți native": sum(counts.values()), "CNOT": counts["CNOT"],
            "porți 1 qubit": sum(counts.values()) - counts["CNOT"], "adâncime": resources.depth,
            "locuri de zgomot": noise_locations(vqc, x, w_probe), "parametri": w_probe.numel(),
            "parametri deplasați (Tracker)": int(round((exec_tracker - 1) / 2)),
            "parametri cu gradient nenul": effective,
            "execuții / exemplu (nominal 1 + 2P)": 1 + 2 * w_probe.numel(),
            "execuții / exemplu (Tracker, parameter shift)": exec_tracker,
        }
    finally:
        ANSATZ_REPS = saved_reps


batch_x, batch_y = torch.tensor(Z_check[:BATCH_SIZE]), torch.tensor(y_bal_check[:BATCH_SIZE], dtype=torch.float64)
grad_x = torch.tensor(Z_check[:256])
structure = pd.DataFrame([circuit_structure(r, torch.tensor(Z_check[0]), batch_x, batch_y, grad_x)
                          for r in [EXP3_ANSATZ_REPS, ANSATZ_REPS]], index=[EXP3_LABEL, EXP7_LABEL])
assert ANSATZ_REPS == 2, "ANSATZ_REPS trebuie restaurat"
for label, reps in [(EXP3_LABEL, EXP3_ANSATZ_REPS), (EXP7_LABEL, ANSATZ_REPS)]:
    for key, value in expected_structure(reps).items():
        assert structure.loc[label, key] == value, f"{label}: {key} = {structure.loc[label, key]}, așteptat {value}"

exp3_saved = pd.read_csv(EXP3_DIR / "circuit_structure.csv", index_col=0).loc["Exp. 3 (ZZ ×1)"]
for key in ["porți native", "CNOT", "adâncime", "locuri de zgomot", "parametri", "execuții / exemplu (Tracker, parameter shift)"]:
    assert structure.loc[EXP3_LABEL, key] == exp3_saved[key], f"Exp. 3 {key}: {structure.loc[EXP3_LABEL, key]} vs salvat {exp3_saved[key]}"

N_PARAMS = int(structure.loc[EXP7_LABEL, "parametri"])
EXECUTIONS_PER_SAMPLE = 1 + 2 * N_PARAMS
EXP3_N_PARAMS = int(structure.loc[EXP3_LABEL, "parametri"])
assert N_PARAMS == 24 and EXP3_N_PARAMS == 32
print("✅ structură: numărătorile coincid cu formula pentru ambele circuite; Exp. 3 coincide cu result_exp3/circuit_structure.csv")
display(structure.T)

## 4. Antrenare fără zgomot (de la zero) și salvarea ponderilor

Pentru fiecare seed: `undersample` 1:1 → `AngleEncoder(4, seed)` → `init_weights(seed)` → `train_vqc_run`, 30 de epoci, epoca aleasă după AUPRC pe validation fără zgomot. Ponderile epocii alese, ponderile inițiale și istoricul (loss, AUPRC și timp pe epocă) se salvează în `results/result_exp7/weights/seed_<s>.pt`, iar un rezumat per seed în `train_runs.jsonl`, **imediat după fiecare seed**. La o rulare ulterioară, ponderile salvate se încarcă și se verifică (AUPRC recalculat), fără reantrenare.

In [ ]:
def load_jsonl(path, required=False):
    """Înregistrările din path. Un fișier obligatoriu lipsă oprește execuția, în loc să întoarcă o listă goală."""
    if not path.exists():
        if required:
            raise FileNotFoundError(f"fișier obligatoriu lipsă: {path}")
        return []
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]


def to_builtin(value):
    if isinstance(value, dict):
        return {k: to_builtin(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_builtin(v) for v in value]
    if isinstance(value, np.generic):
        return value.item()
    return value


def append_jsonl(path, record):
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(to_builtin(record), ensure_ascii=False) + "\n")


config = {
    "experiment": EXPERIMENT_NAME, "status": "exploratory (validation only; test results were already consulted earlier in the project)",
    "change_vs_exp3": "ANSATZ_REPS 3 -> 2 (only change)", "reference": "Exp. 3 (results/result_exp3, read only)",
    "feature_map_reps": FEATURE_MAP_REPS, "ansatz_reps": ANSATZ_REPS, "n_qubits": N_QUBITS, "n_params": N_PARAMS,
    "seeds": SEEDS, "noise_levels": P_ALL, "epochs": EPOCHS, "batch_size": BATCH_SIZE, "learning_rate": LEARNING_RATE,
    "optimizer": "Adam", "loss": "BCELoss", "init": "init_weights(seed) (= first 3 layers of Exp. 3's initial weights)",
    "shuffle": "torch.Generator().manual_seed(seed) (verified identical to the official global randperm after set_seed)",
    "balancing": "undersample(X_train, y_train, seed), ratio 1:1, 295 fraud + 295 legit = 590 per seed (verified)",
    "encoder": "AngleEncoder(4, seed) fitted on the 1:1 set", "epoch_selection": "best validation AUPRC at p = 0",
    "noise_model": "noisy_qnode: NATIVE_GATES decomposition, DepolarizingChannel(p) after every gate on every wire, default.mixed",
    "threshold": "max F1 on validation scores at the same p", "evaluation_set": "validation (test not used)",
    "split": "split_indices.npz (verified against official reconstruction)", "shots_per_execution": SHOTS_PER_EXECUTION,
    "official_configuration_confirmed": {
        "source": "notebooks/task1_guided_vqc_fraud_detection.ipynb + docs/Fraud_on_a_Quantum_Budget_Task.pdf",
        "dataset": "284,807 transactions, 492 fraud (OpenML 42175 / Kaggle)",
        "split": "60/20/20 stratified, seed 42; train 170,883 (295 fraud), val 56,962 (99), test 56,962 (98)",
        "undersampling": "all training frauds + as many legit drawn without replacement, rng = default_rng(seed), then permuted",
        "training_examples": 590, "n_qubits": 4,
        "feature_map": "ZZ: H + RZ(2 x_i) per qubit; per pair i<j: CNOT, RZ(2(pi-x_i)(pi-x_j)), CNOT; FEATURE_MAP_REPS = 2",
        "ansatz": "EfficientSU2-style: RY+RZ per qubit, linear CNOT ladder, x ANSATZ_REPS + final rotation layer; params = (ANSATZ_REPS+1)*4*2; ANSATZ_REPS = 3 (32 params)",
        "training": "Adam, lr 0.05, batch 32, BCELoss, 30 epochs, best validation AUPRC epoch",
        "noise": "depolarizing p = 0.001, 0.01, 0.05 after every native gate on every qubit, default.mixed",
        "metrics": "AUPRC (primary, p = 0.01 and 0.05), AUROC, Spearman rho, precision, recall, F1, MCC; threshold max F1 on validation at the same p; mean +- std over >= 5 seeds",
        "cost": "training = epochs * 590 * (1 + 2P); validation = epochs * n_val; 1024 shots per execution",
    },
    "differences_vs_official": ["FEATURE_MAP_REPS 2 -> 1 (Exp. 3 change, kept)", "ANSATZ_REPS 3 -> 2 (Exp. 7 change)",
                                "dedicated shuffle generator (same permutations as official)", "evaluated on validation, not test"],
    "differences_vs_exp3": ["ANSATZ_REPS 3 -> 2", "n_params 32 -> 24"],
    "versions": {"pennylane": qml.__version__, "torch": torch.__version__, "sklearn": sklearn.__version__, "numpy": np.__version__},
}
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
CONFIG_PATH = RESULTS_DIR / "config.json"
TRAIN_RUNS_PATH, VAL_RUNS_PATH = RESULTS_DIR / "train_runs.jsonl", RESULTS_DIR / "val_runs.jsonl"
if CONFIG_PATH.exists():
    previous = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
    current = json.loads(json.dumps(to_builtin(config)))
    if {k: v for k, v in previous.items() if k != "versions"} != {k: v for k, v in current.items() if k != "versions"}:
        raise RuntimeError(f"{CONFIG_PATH} conține altă configurație. Nu suprascriu: mută directorul sau schimbă RESULTS_DIR.")
else:
    CONFIG_PATH.write_text(json.dumps(to_builtin(config), indent=2, ensure_ascii=False), encoding="utf-8")

train_done = {r["seed"]: r for r in load_jsonl(TRAIN_RUNS_PATH)}
models = {}
for seed in SEEDS:
    X_s, y_s = undersample(X_train, y_train, seed)
    encoder = AngleEncoder(N_QUBITS, seed).fit(X_s)
    Z_tr, Z_val = encoder.transform(X_s), encoder.transform(X_val)
    path = WEIGHTS_DIR / f"seed_{seed}.pt"
    if path.exists() and seed in train_done:
        saved = torch.load(path, weights_only=True)
        source = "încărcat"
    else:
        initial = init_weights(seed)
        run = train_vqc_run(Z_tr, y_s, Z_val, y_val, initial, seed)
        saved = {"weights": run["weights"], "best_epoch": run["best_epoch"], "best_val_ap": run["best_val_ap"],
                 "history": run["history"], "initial_weights": initial.detach().clone()}
        torch.save(saved, path)
        append_jsonl(TRAIN_RUNS_PATH, {"seed": seed, "best_epoch": run["best_epoch"], "val AUPRC (p = 0)": run["best_val_ap"],
                                       "epoci executate": len(run["history"]), "exemple train": len(y_s), "fraude train": int(y_s.sum()),
                                       "legitime train": int((y_s == 0).sum()), "pași Adam": run["adam_steps"],
                                       "timp antrenare [s]": run["time_train_s"],
                                       "train loss per epocă": [h["train loss"] for h in run["history"]],
                                       "val AUPRC per epocă": [h["validation AUPRC"] for h in run["history"]]})
        train_done[seed] = load_jsonl(TRAIN_RUNS_PATH, required=True)[-1]
        source = f"antrenat și salvat ({run['time_train_s']:.0f} s)"
    assert saved["weights"].shape == (ANSATZ_REPS + 1, N_QUBITS, 2)
    assert np.isclose(average_precision_score(y_val, predict(Z_val, saved["weights"])), saved["best_val_ap"]), f"seed {seed}: ponderile nu reproduc AUPRC-ul salvat"
    assert len(saved["history"]) == EPOCHS and saved["best_epoch"] == 1 + int(np.argmax([h["validation AUPRC"] for h in saved["history"]]))
    assert train_done[seed]["pași Adam"] == EPOCHS * math.ceil(590 / BATCH_SIZE) and train_done[seed]["exemple train"] == 590
    models[seed] = {"weights": saved["weights"], "Z_val": Z_val, "best_epoch": saved["best_epoch"], "history": saved["history"]}
    print(f"seed {seed}: {source}   epoca aleasă {saved['best_epoch']:2d}   val AUPRC (p = 0) {saved['best_val_ap']:.4f}")
print(f"✅ 5 modele Exp. 7: ponderi salvate, AUPRC reprodus; {EPOCHS} epoci × {math.ceil(590 / BATCH_SIZE)} pași = {EPOCHS * math.ceil(590 / BATCH_SIZE)} pași Adam per seed")

## 5. Evaluare sub zgomot pe validation

Identic cu Exp. 3: pentru fiecare seed și p, scorurile validării (fără zgomot cu `predict`, cu zgomot cu `noisy_predict`), pragul max-F1 pe validation **la același p**, metricile la acest prag și Spearman ρ față de scorurile fără zgomot. Fiecare (seed, p) terminat se salvează imediat în `val_runs.jsonl`.

Precision / recall / F1 / MCC sunt optimiste (prag ales pe același set), exact ca în Exp. 3, deci comparația rămâne corectă.

In [ ]:
w0, Zv0 = models[SEEDS[0]]["weights"], models[SEEDS[0]]["Z_val"]   # sanity: default.mixed la p = 0 = state-vector
sanity = np.abs(noisy_predict(Zv0[:2000], w0, 0.0) - predict(Zv0[:2000], w0)).max()
assert sanity < 1e-6, sanity
print(f"default.mixed la p = 0 vs default.qubit (2000 tranzacții): diferență maximă {sanity:.1e}")

val_done = {(r["seed"], r["p"]) for r in load_jsonl(VAL_RUNS_PATH)}
print(f"treceri zgomotoase de făcut pe validation: {sum((s, p) not in val_done for s in SEEDS for p in NOISE_LEVELS)} din {len(SEEDS) * len(NOISE_LEVELS)}")
for seed in SEEDS:
    m = models[seed]
    clean_val = predict(m["Z_val"], m["weights"])
    for p in P_ALL:
        if (seed, p) in val_done:
            continue
        t0 = time.perf_counter()
        scores = clean_val if p == 0.0 else noisy_predict(m["Z_val"], m["weights"], p)
        elapsed = time.perf_counter() - t0
        threshold = choose_threshold(y_val, scores)
        metrics = evaluate(y_val, scores, threshold)
        record = {"seed": seed, "p": p, **metrics, "Spearman rho": spearmanr(clean_val, scores).statistic,
                  "threshold": float(threshold), "timp scorare [s]": elapsed}
        append_jsonl(VAL_RUNS_PATH, record)
        val_done.add((seed, p))
        print(f"seed {seed}  p = {p:<5}  val AUPRC {metrics['AUPRC']:.3f}  F1 {metrics['F1']:.3f}  TP {metrics['TP']:2d}  FP {metrics['FP']:3d}  rho {record['Spearman rho']:.4f}  ({elapsed:.0f} s)")

## 6. Rezultate Exp. 7 și comparația cu Exp. 3

Rezultatele Exp. 3 sunt citite din `results/result_exp3/val_runs.jsonl` (doar citit). Toate valorile sunt pe **validation** (99 de fraude). Medie ± std (ddof = 1) pe 5 seeds; diferențele sunt **pereche**: Exp. 7 − Exp. 3 pe același seed.

In [ ]:
METRICS = ["AUPRC", "AUROC", "Spearman rho", "precision", "recall", "F1", "MCC", "TP", "FP"]


def load_results(path):
    records = [r for r in load_jsonl(path, required=True) if r["seed"] in SEEDS and r["p"] in P_ALL]
    keys = [(r["seed"], r["p"]) for r in records]
    missing = sorted({(s, p) for s in SEEDS for p in P_ALL} - set(keys))
    assert not missing, f"lipsesc rezultate (seed, p) în {path}: {missing}"
    assert len(keys) == len(set(keys)), f"rezultate duplicate în {path}"
    return pd.DataFrame(records).sort_values(["p", "seed"]).reset_index(drop=True)


def digits_for(metric):
    return 4 if metric == "Spearman rho" else 1 if metric in ("TP", "FP") else 3


def mean_std(values, digits=3):
    values = np.asarray(values, dtype=float)
    return f"{values.mean():.{digits}f} ± {values.std(ddof=1):.{digits}f}"


exp7 = load_results(VAL_RUNS_PATH)
exp3 = load_results(EXP3_DIR / "val_runs.jsonl")
summary = pd.DataFrame([
    {"p": p, **{m: mean_std(exp7.loc[exp7["p"] == p, m], digits_for(m)) for m in METRICS},
     "prag": mean_std(exp7.loc[exp7["p"] == p, "threshold"])}
    for p in P_ALL
]).set_index("p")
summary.to_csv(RESULTS_DIR / "val_summary.csv")
print(f"Exp. 7 (ansatz ×2), medie ± std pe {len(SEEDS)} seeds — validation")
display(summary)


def paired(metric, p):
    a = exp3[exp3["p"] == p].set_index("seed").loc[SEEDS, metric].astype(float)
    b = exp7[exp7["p"] == p].set_index("seed").loc[SEEDS, metric].astype(float)
    return a, b, b - a


comparison = []
for p in P_ALL:
    for m in METRICS:
        a, b, d = paired(m, p)
        dg = digits_for(m)
        comparison.append({"p": p, "metrică": m, EXP3_LABEL: mean_std(a, dg), EXP7_LABEL: mean_std(b, dg),
                           "diferență pereche (Exp. 7 − Exp. 3)": f"{d.mean():+.{dg}f} ± {d.std(ddof=1):.{dg}f}",
                           "seeds Exp. 7 > Exp. 3": int((d > 0).sum()), "seeds Exp. 7 < Exp. 3": int((d < 0).sum()),
                           "seeds egal": int((d == 0).sum())})
comparison = pd.DataFrame(comparison).set_index(["p", "metrică"])
comparison.to_csv(RESULTS_DIR / "comparison_vs_exp3.csv")
print(f"Comparație {EXP3_LABEL} vs {EXP7_LABEL}, medie ± std pe {len(SEEDS)} seeds — validation")
display(comparison)

per_seed = []
for p in P_ALL:
    for seed in SEEDS:
        row = {"p": p, "seed": seed}
        for m, short in [("AUPRC", "AUPRC"), ("Spearman rho", "rho"), ("F1", "F1"), ("MCC", "MCC"), ("TP", "TP"), ("FP", "FP")]:
            a, b, d = paired(m, p)
            row.update({f"{short} Exp. 3": a[seed], f"{short} Exp. 7": b[seed], f"Δ {short}": d[seed]})
        per_seed.append(row)
per_seed = pd.DataFrame(per_seed)
per_seed.to_csv(RESULTS_DIR / "paired_differences.csv", index=False)
print("Diferențe pereche per seed (Exp. 7 − Exp. 3), p = 0.01 și p = 0.05")
display(per_seed[per_seed["p"].isin([0.01, 0.05])].set_index(["p", "seed"])[
    ["AUPRC Exp. 3", "AUPRC Exp. 7", "Δ AUPRC", "Δ rho", "Δ F1", "Δ MCC", "Δ TP", "Δ FP"]].round(4))

### Degradare sub zgomot, fraude detectate și alarme false

- **Degradarea AUPRC** = AUPRC(p = 0) − AUPRC(p = 0.05), per seed: cât pierde fiecare model din cauza zgomotului (mai mic = mai robust).
- **Stabilitatea** pe seeds = std-ul AUPRC la fiecare p.
- Fraude detectate (TP, din 99) și alarme false (FP) la pragul max-F1 de pe validation.

In [ ]:
degradation = []
for seed in SEEDS:
    row = {"seed": seed}
    for name, res in [("Exp. 3", exp3), ("Exp. 7", exp7)]:
        r = res[res["seed"] == seed].set_index("p")
        row[f"{name}: AUPRC p=0"] = r.loc[0.0, "AUPRC"]
        row[f"{name}: AUPRC p=0.05"] = r.loc[0.05, "AUPRC"]
        row[f"{name}: degradare 0→0.05"] = r.loc[0.0, "AUPRC"] - r.loc[0.05, "AUPRC"]
        row[f"{name}: degradare 0→0.01"] = r.loc[0.0, "AUPRC"] - r.loc[0.01, "AUPRC"]
    degradation.append(row)
degradation = pd.DataFrame(degradation).set_index("seed")
deg_summary = pd.DataFrame({c: [mean_std(degradation[c])] for c in degradation.columns}, index=["medie ± std"]).T
d_diff = degradation["Exp. 7: degradare 0→0.05"] - degradation["Exp. 3: degradare 0→0.05"]
print("Degradarea AUPRC, per seed")
display(degradation.round(4))
display(deg_summary)
print(f"diferență pereche a degradării 0→0.05 (Exp. 7 − Exp. 3): {d_diff.mean():+.3f} ± {d_diff.std(ddof=1):.3f}; "
      f"Exp. 7 se degradează mai puțin la {int((d_diff < 0).sum())} / {len(SEEDS)} seeds")

detect = pd.DataFrame([
    {"p": p, "model": name, "fraude detectate (TP, din 99)": mean_std(res.loc[res["p"] == p, "TP"], 1),
     "fraude ratate (FN)": mean_std(res.loc[res["p"] == p, "FN"], 1), "alarme false (FP)": mean_std(res.loc[res["p"] == p, "FP"], 1),
     "recall": mean_std(res.loc[res["p"] == p, "recall"]), "precision": mean_std(res.loc[res["p"] == p, "precision"]),
     "std AUPRC pe seeds": f"{res.loc[res['p'] == p, 'AUPRC'].std(ddof=1):.3f}",
     "Spearman rho": mean_std(res.loc[res["p"] == p, "Spearman rho"], 4)}
    for p in P_ALL for name, res in [("Exp. 3", exp3), ("Exp. 7", exp7)]
]).set_index(["p", "model"])
print("Fraude detectate și alarme false pe validation (pragul max-F1 la același p)")
display(detect)

## 7. Costul circuitului: Exp. 3 vs Exp. 7

Separat, patru lucruri:
1. **Structura circuitului** (secțiunea 3): porți, CNOT, adâncime, locuri de zgomot, parametri. Determină cât zgomot acumulează un shot.
2. **Estimare hardware (formula Părții 8)**: antrenare = epoci × 590 × (1 + 2P); validare după fiecare epocă = epoci × n_val; o trecere finală pe validation la p = 0 și câte una pentru fiecare p cu zgomot; testul doar planificat (neexecutat în Exp. 7). Shots = execuții × 1024.
3. **Estimare hardware cu `qml.Tracker`**: antrenare = epoci × 590 × (execuții măsurate pe exemplu), adică doar parametrii din conul de lumină al qubitului 0.
4. **Ce s-a simulat efectiv** (circuite evaluate pe simulator, gradient prin backprop, nu parameter shift) și **timpul real de simulare**. Acestea nu sunt cost hardware.

In [ ]:
n_val = len(val_idx)
n_test = len(test_idx)
exp3_train = {r["seed"]: r for r in load_jsonl(EXP3_DIR / "train_runs.jsonl", required=True)}
exp7_train = {r["seed"]: r for r in load_jsonl(TRAIN_RUNS_PATH, required=True)}
exp3_ledger_saved = pd.read_csv(EXP3_DIR / "cost_ledger.csv", index_col=0)


def ledger_rows(n_params, exec_tracker, epochs):
    return [
        ("1a. antrenare: forward + parameter shift (formula 1 + 2P)", "estimare hardware", epochs * 590 * (1 + 2 * n_params)),
        ("1b. antrenare: parameter shift doar în conul de lumină (Tracker)", "estimare hardware", int(round(epochs * 590 * exec_tracker))),
        ("2. validare după fiecare epocă, p = 0", "estimare hardware", epochs * n_val),
        ("3a. validation p = 0, ponderile alese (o dată)", "estimare hardware", n_val),
        ("3b. validation sub zgomot, p = 0.001 / 0.01 / 0.05", "estimare hardware", len(NOISE_LEVELS) * n_val),
        ("4. test, 4 niveluri de p", "planificat, neexecutat în Exp. 7", len(P_ALL) * n_test),
    ]


assert {r["epoci executate"] for r in exp3_train.values()} == {r["epoci executate"] for r in exp7_train.values()} == {EPOCHS}
rows3 = ledger_rows(EXP3_N_PARAMS, structure.loc[EXP3_LABEL, "execuții / exemplu (Tracker, parameter shift)"], EPOCHS)
rows7 = ledger_rows(N_PARAMS, structure.loc[EXP7_LABEL, "execuții / exemplu (Tracker, parameter shift)"], EPOCHS)
assert rows3[0][2] == exp3_ledger_saved.iloc[0]["execuții / seed, Exp. 3 (ZZ ×1)"] == 1_150_500, "formula pentru Exp. 3 trebuie să reproducă ledger-ul salvat"

n_s = len(SEEDS)
ledger = pd.DataFrame([
    {"componentă": name, "tip": kind, "execuții / seed, Exp. 3": a, "execuții / seed, Exp. 7": b,
     "diferență / seed (Exp. 7 − Exp. 3)": b - a, "reducere": f"{1 - b / a:.0%}" if a else "",
     f"execuții {n_s} seeds, Exp. 7": b * n_s, f"shots {n_s} seeds, Exp. 3": a * n_s * SHOTS_PER_EXECUTION,
     f"shots {n_s} seeds, Exp. 7": b * n_s * SHOTS_PER_EXECUTION}
    for (name, kind, a), (_, _, b) in zip(rows3, rows7)
]).set_index("componentă")
ledger.to_csv(RESULTS_DIR / "cost_ledger.csv")
print(f"Estimare hardware: execuții și shots ({SHOTS_PER_EXECUTION} shots / execuție)")
display(ledger.map(lambda v: f"{v:,}" if isinstance(v, (int, np.integer)) else v))

formula_rows = [0, 2, 3, 4]   # antrenare (formula) + validare: totalul cerut de Task 2, fără test
tot3, tot7 = sum(rows3[i][2] for i in formula_rows), sum(rows7[i][2] for i in formula_rows)
print(f"Antrenare + validare per seed (formula): Exp. 3 {tot3:,}   Exp. 7 {tot7:,}   reducere {1 - tot7 / tot3:.1%}")
print(f"Doar antrenare per seed (formula): Exp. 3 {rows3[0][2]:,}   Exp. 7 {rows7[0][2]:,}   reducere {1 - rows7[0][2] / rows3[0][2]:.1%}")

exp7_noisy_time = exp7.loc[exp7["p"] > 0, "timp scorare [s]"].sum()
exp3_noisy_time = exp3.loc[exp3["p"] > 0, "timp scorare [s]"].sum()
simulated = pd.DataFrame([
    {"ce": "circuite simulate la antrenare (forward, gradient prin backprop)", "Exp. 3": EPOCHS * 590, "Exp. 7": EPOCHS * 590},
    {"ce": "circuite simulate la validarea pe epoci (default.qubit)", "Exp. 3": EPOCHS * n_val, "Exp. 7": EPOCHS * n_val},
    {"ce": "circuite simulate cu matrice de densitate (3 niveluri de p)", "Exp. 3": 3 * n_val, "Exp. 7": 3 * n_val},
    {"ce": "timp antrenare, 5 seeds [min]", "Exp. 3": sum(exp3_train[s]["timp antrenare [s]"] for s in SEEDS) / 60,
     "Exp. 7": sum(exp7_train[s]["timp antrenare [s]"] for s in SEEDS) / 60},
    {"ce": "timp treceri zgomotoase pe validation, 5 seeds [min]", "Exp. 3": exp3_noisy_time / 60, "Exp. 7": exp7_noisy_time / 60},
]).set_index("ce")
print("Simulări executate efectiv per seed și timp real (informativ, depinde de mașină; NU este cost hardware):")
display(simulated.round(2))
simulated.to_csv(RESULTS_DIR / "simulator_time.csv")

circuit_comparison = structure.T.copy()
circuit_comparison["diferență (Exp. 7 − Exp. 3)"] = circuit_comparison[EXP7_LABEL] - circuit_comparison[EXP3_LABEL]
circuit_comparison["reducere"] = [f"{1 - b / a:.0%}" if a else "" for a, b in zip(circuit_comparison[EXP3_LABEL], circuit_comparison[EXP7_LABEL])]
for label, row in [("execuții antrenare / seed (formula)", (rows3[0][2], rows7[0][2])), ("execuții antrenare / seed (Tracker)", (rows3[1][2], rows7[1][2])),
                   ("execuții antrenare + validare / seed (formula)", (tot3, tot7)),
                   ("shots antrenare + validare / seed (formula)", (tot3 * SHOTS_PER_EXECUTION, tot7 * SHOTS_PER_EXECUTION)),
                   ("timp antrenare 5 seeds [s]", (simulated.loc["timp antrenare, 5 seeds [min]", "Exp. 3"] * 60, simulated.loc["timp antrenare, 5 seeds [min]", "Exp. 7"] * 60)),
                   ("timp treceri zgomotoase 5 seeds [s]", (exp3_noisy_time, exp7_noisy_time))]:
    a, b = row
    circuit_comparison.loc[label] = [a, b, b - a, f"{1 - b / a:.0%}"]
circuit_comparison.to_csv(RESULTS_DIR / "circuit_comparison.csv")
print("Comparație structurală și de cost")
display(circuit_comparison)

## 8. Grafice

Toate pe validation. Albastru = Exp. 3, portocaliu = Exp. 7; punctele gri sunt seed-urile individuale. Fiecare grafic se salvează în `results/result_exp7/plots/`.

In [ ]:
xpos = np.arange(len(P_ALL))
xlabels = ["0", *[str(p) for p in NOISE_LEVELS]]
MODELS = [(exp3, EXP3_COLOR, "o", EXP3_LABEL, -0.08), (exp7, EXP7_COLOR, "s", EXP7_LABEL, 0.08)]


def metric_vs_p(col, title, fname, ylabel):
    fig, ax = plt.subplots(figsize=(6.5, 4.2))
    for res, color, marker, label, dx in MODELS:
        groups = [res.loc[res["p"] == p, col].to_numpy(dtype=float) for p in P_ALL]
        for i, g in enumerate(groups):
            ax.scatter(np.full(len(g), xpos[i] + dx), g, s=18, color=SEED_DOT_COLOR, alpha=0.6, zorder=2)
        ax.errorbar(xpos + dx, [g.mean() for g in groups], yerr=[g.std(ddof=1) for g in groups], color=color, marker=marker,
                    markersize=8, capsize=4, label=label, zorder=3)
    ax.set_xticks(xpos, xlabels)
    ax.set_xlabel("probabilitate depolarizantă p per poartă")
    ax.set_ylabel(ylabel)
    ax.set_title(f"{title} · validation · medie ± std, 5 seeds")
    ax.legend(loc="lower left")
    plt.tight_layout()
    fig.savefig(PLOTS_DIR / fname, dpi=130)
    plt.show()


metric_vs_p("AUPRC", "AUPRC vs p", "01_auprc_vs_p.png", "AUPRC")
metric_vs_p("F1", "F1 (prag max-F1 la același p) vs p", "02_f1_vs_p.png", "F1")
metric_vs_p("Spearman rho", "Spearman ρ (zgomot vs fără zgomot) vs p", "03_spearman_vs_p.png", "Spearman ρ")

# 4. AUPRC pe fiecare seed, câte un panou pe nivel de zgomot
fig, axes = plt.subplots(1, len(P_ALL), figsize=(15, 3.8), sharey=True)
sx = np.arange(len(SEEDS))
for ax, p in zip(axes, P_ALL):
    a, b, _ = paired("AUPRC", p)
    for s_i in sx:
        ax.plot([s_i - 0.12, s_i + 0.12], [a.iloc[s_i], b.iloc[s_i]], color="0.75", linewidth=1, zorder=1)
    ax.scatter(sx - 0.12, a, s=64, color=EXP3_COLOR, marker="o", label=EXP3_LABEL, zorder=3, edgecolors="white", linewidths=1.5)
    ax.scatter(sx + 0.12, b, s=64, color=EXP7_COLOR, marker="s", label=EXP7_LABEL, zorder=3, edgecolors="white", linewidths=1.5)
    ax.set_xticks(sx, [f"s{s}" for s in SEEDS])
    ax.set_title(f"p = {p}")
axes[0].set_ylabel("AUPRC pe validation")
axes[0].legend(loc="lower left", fontsize=8)
fig.suptitle("AUPRC per seed (aceleași date, encoder și ordine a batch-urilor în fiecare pereche)", fontsize=11)
plt.tight_layout()
fig.savefig(PLOTS_DIR / "04_auprc_per_seed.png", dpi=130)
plt.show()

# 5. diferențe pereche de AUPRC
fig, ax = plt.subplots(figsize=(6.5, 4.2))
for seed in SEEDS:
    d = per_seed[per_seed["seed"] == seed].set_index("p").loc[P_ALL, "Δ AUPRC"]
    ax.plot(xpos, d, color=SEED_DOT_COLOR, marker="o", markersize=5, linewidth=1, alpha=0.8, zorder=2)
    ax.annotate(f"s{seed}", (xpos[-1], d.iloc[-1]), textcoords="offset points", xytext=(6, 0), fontsize=8, color="0.35", va="center")
mean_d = per_seed.groupby("p")["Δ AUPRC"].mean().loc[P_ALL]
ax.plot(xpos, mean_d, color=EXP7_COLOR, marker="s", markersize=8, label="medie pe 5 seeds", zorder=3)
ax.axhline(0, color="0.3", linestyle="--", linewidth=1)
ax.set_xticks(xpos, xlabels)
ax.set_xlabel("probabilitate depolarizantă p per poartă")
ax.set_ylabel("Δ AUPRC")
ax.set_title("Δ AUPRC pereche (Exp. 7 − Exp. 3; > 0 = Exp. 7 mai bun)")
ax.legend(loc="lower left")
plt.tight_layout()
fig.savefig(PLOTS_DIR / "05_paired_auprc_differences.png", dpi=130)
plt.show()

# 6. fraude detectate și alarme false
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
width = 0.36
for ax, col, title in zip(axes, ["TP", "FP"], ["Fraude detectate (TP, din 99)", "Alarme false (FP)"]):
    for (res, color, marker, label, _), off in zip(MODELS, [-width / 2 - 0.01, width / 2 + 0.01]):
        groups = [res.loc[res["p"] == p, col].to_numpy(dtype=float) for p in P_ALL]
        bars = ax.bar(xpos + off, [g.mean() for g in groups], width, yerr=[g.std(ddof=1) for g in groups], color=color,
                      capsize=3, label=label, error_kw={"elinewidth": 1, "ecolor": "0.3"})
        ax.bar_label(bars, labels=[f"{g.mean():.1f}" for g in groups], padding=2, fontsize=8, color="0.25")
    ax.set_xticks(xpos, xlabels)
    ax.set_xlabel("probabilitate depolarizantă p per poartă")
    ax.set_title(f"{title} · medie ± std, 5 seeds")
axes[0].legend(loc="lower left")
plt.tight_layout()
fig.savefig(PLOTS_DIR / "06_frauds_and_false_alarms.png", dpi=130)
plt.show()

# 7. comparație structurală și de cost
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
keys = ["porți native", "CNOT", "adâncime", "locuri de zgomot", "parametri", "parametri cu gradient nenul"]
kx = np.arange(len(keys))
for (label, color), off in zip([(EXP3_LABEL, EXP3_COLOR), (EXP7_LABEL, EXP7_COLOR)], [-0.19, 0.19]):
    vals = [structure.loc[label, k] for k in keys]
    bars = axes[0].bar(kx + off, vals, 0.36, color=color, label=label)
    axes[0].bar_label(bars, labels=[f"{v:g}" for v in vals], padding=2, fontsize=8, color="0.25")
axes[0].set_xticks(kx, keys, rotation=20, ha="right")
axes[0].set_title("Structura circuitului")
axes[0].legend(loc="upper right")
cost_keys = [("antrenare\n(formula)", 0), ("antrenare\n(Tracker)", 1), ("validare pe\nepoci", 2)]
cx = np.arange(len(cost_keys))
for (rows, label, color), off in zip([(rows3, EXP3_LABEL, EXP3_COLOR), (rows7, EXP7_LABEL, EXP7_COLOR)], [-0.19, 0.19]):
    vals = [rows[i][2] / 1e6 for _, i in cost_keys]
    bars = axes[1].bar(cx + off, vals, 0.36, color=color, label=label)
    axes[1].bar_label(bars, labels=[f"{v:.2f}M" for v in vals], padding=2, fontsize=8, color="0.25")
axes[1].set_xticks(cx, [k for k, _ in cost_keys])
axes[1].set_ylabel("milioane de execuții / seed")
axes[1].set_title(f"Estimare hardware ({SHOTS_PER_EXECUTION} shots / execuție)")
plt.tight_layout()
fig.savefig(PLOTS_DIR / "07_structure_and_cost.png", dpi=130)
plt.show()

# 8. evoluția antrenării
exp3_hist = {s: torch.load(EXP3_DIR / "weights" / f"seed_{s}.pt", weights_only=True)["history"] for s in SEEDS}
exp7_hist = {s: models[s]["history"] for s in SEEDS}
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
epochs_x = np.arange(1, EPOCHS + 1)
for hist, color, label in [(exp3_hist, EXP3_COLOR, EXP3_LABEL), (exp7_hist, EXP7_COLOR, EXP7_LABEL)]:
    for ax, col in zip(axes, ["train loss", "validation AUPRC"]):
        curves = np.array([[h[col] for h in hist[s]] for s in SEEDS])
        for c in curves:
            ax.plot(epochs_x, c, color=color, linewidth=0.8, alpha=0.3)
        ax.plot(epochs_x, curves.mean(axis=0), color=color, label=f"{label}, medie")
for s in SEEDS:   # epocile alese
    axes[1].scatter(models[s]["best_epoch"], exp7_hist[s][models[s]["best_epoch"] - 1]["validation AUPRC"], color=EXP7_COLOR, s=40, marker="s", zorder=3, edgecolors="white")
    b3 = int(np.argmax([h["validation AUPRC"] for h in exp3_hist[s]]))
    axes[1].scatter(b3 + 1, exp3_hist[s][b3]["validation AUPRC"], color=EXP3_COLOR, s=40, marker="o", zorder=3, edgecolors="white")
axes[0].set_title("Training loss (BCE) pe epocă")
axes[1].set_title("AUPRC pe validation (p = 0) pe epocă; marcaj = epoca aleasă")
for ax in axes:
    ax.set_xlabel("epocă")
    ax.legend()
plt.tight_layout()
fig.savefig(PLOTS_DIR / "08_training_curves.png", dpi=130)
plt.show()
print("grafice salvate:", sorted(p.name for p in PLOTS_DIR.glob("*.png")))

## 9. Tabel final

Medie ± std pe 5 seeds, **validation**. Pentru Exp. 3 se folosesc rezultatele salvate (neschimbate).

In [ ]:
final_rows = []
for p in P_ALL:
    for name, res in [("Exp. 3 (ansatz ×3)", exp3), ("Exp. 7 (ansatz ×2)", exp7)]:
        r = res[res["p"] == p]
        final_rows.append({"p": p, "model": name, **{m: mean_std(r[m], digits_for(m)) for m in ["AUPRC", "AUROC", "Spearman rho", "F1", "MCC", "recall", "TP", "FP"]}})
final = pd.DataFrame(final_rows).set_index(["p", "model"])
display(final)
wins = pd.DataFrame([{"p": p, **{m: f"{int((paired(m, p)[2] > 0).sum())} / {len(SEEDS)}" for m in ["AUPRC", "Spearman rho", "F1", "MCC"]}} for p in P_ALL]).set_index("p")
print("Seeds la care Exp. 7 > Exp. 3")
display(wins)

## 10. Concluzii (validation, exploratoriu)

**H1 respinsă pentru performanță; confirmată doar parțial pentru robustețe și cost.**

- **AUPRC:** Exp. 7 e mai slab la toate nivelurile de zgomot: −0.074 la p = 0, **−0.081 ± 0.051 la p = 0.01** (Exp. 3 câștigă la 4/5 seeds) și **−0.067 ± 0.037 la p = 0.05** (Exp. 3 câștigă la 5/5 seeds). Pierderea vine din capacitate, nu din zgomot: deja la p = 0 Exp. 7 pierde la toate cele 5 seeds, iar loss-ul de antrenare se oprește la ~0.42 față de ~0.36 (subadaptare). Doar 11 parametri ajung efectiv la qubitul 0, față de 19.
- **Robustețe:** ranking-ul e mai stabil (Spearman ρ la p = 0.05: 0.978 ± 0.013 față de 0.875 ± 0.110, mai mare la 4/5 seeds), iar AUPRC variază mai puțin între seeds la p = 0.05 (std 0.013 față de 0.039). Degradarea AUPRC 0 → 0.05 nu diferă însă semnificativ (0.033 ± 0.036 față de 0.041 ± 0.047; diferență pereche −0.007 ± 0.068, 2/5 seeds în favoarea Exp. 7).
- **Decizii (prag max-F1):** la p ≤ 0.01 Exp. 7 detectează cu ~7–9 fraude mai puțin din 99 și dă cu ~4–7 alarme false mai puțin; la p = 0.05 detectează aproape la fel (64.4 față de 63.6), dar cu mai multe alarme false (30.0 față de 24.6). F1 și MCC sunt mai mici la fiecare p.
- **Cost:** −11 porți, −3 CNOT, −4 adâncime, −14 locuri de zgomot, 24 de parametri în loc de 32. Antrenarea pe hardware scade cu 25% (formula) / 39% (Tracker), dar antrenare + validare scade doar cu 9%, pentru că validarea pe fiecare epocă (1.71M execuții) nu depinde de ansatz.

**Verdict:** câștigul de robustețe al rangului și reducerea de cost nu compensează pierderea de ~0.07–0.08 AUPRC, constantă la toate nivelurile de zgomot. **Exp. 3 rămâne soluția principală**; Exp. 7 se raportează ca ablație (rezultat negativ): în acest circuit, zgomotul ansatz-ului nu este factorul limitant, capacitatea este.
